# Preprocessing CHIRPS V3 district rainfall dataset


Merges the per-period CHIRPS V3 district rainfall extracts into one daily
rainfall file, and resolves the duplicated MBALE records left by the zonal-stats
extraction.

**Input:** `dataset/CHIRPS_Elgon/chirps_daily_districts_*.csv` — six files,
`date, district, rain_mean, rain_max, rain_min`, catchment statistics in mm/day.

**Output:** `dataset/chirps_daily_rainfall.csv` — one row per district-day.

| Step | Effect |
| --- | --- |
| 1. Load the six extracts | 102,270 rows |
| 2. Drop Mbale Municipality | 102,270 → 92,043 |
| 3. Verify and write | 92,043 rows, 9 districts |

## 1. Load the extracts

`dataset_path` is resolved against the repository root rather than the kernel's
working directory, which depends on where Jupyter was launched.

In [ ]:
from pathlib import Path

import pandas as pd

CHIRPS_DIR = "dataset/CHIRPS_Elgon"
OUTPUT_PATH = "dataset/chirps_daily_rainfall.csv"

RAIN_COLUMNS = ["rain_mean", "rain_max", "rain_min"]


def find_repo_root(marker: str = "dataset") -> Path:
    """Walk up from the working directory until `marker` is found."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()

files = sorted((REPO_ROOT / CHIRPS_DIR).glob("chirps_daily_districts_*.csv"))
if not files:
    raise FileNotFoundError(f"No CHIRPS extracts found in {REPO_ROOT / CHIRPS_DIR}")

frames = []
for path in files:
    frame = pd.read_csv(path)
    if frame.empty:
        print(f"  skipped {path.name} (header only)")
        continue
    frame["date"] = pd.to_datetime(frame["date"])
    frame["district"] = frame["district"].str.strip().str.upper()
    # Row order within a file matters for identifying the MBALE polygons, so it
    # is preserved explicitly rather than left to depend on later sorting
    frame["source_file"] = path.name
    frame["file_row"] = range(len(frame))
    frames.append(frame)
    print(f"  {path.name:<42} {len(frame):>6} rows  "
          f"{frame['date'].min():%Y-%m-%d}..{frame['date'].max():%Y-%m-%d}")

rainfall = pd.concat(frames, ignore_index=True)

print(f"\nloaded {len(rainfall):,} rows from {len(frames)} files")
print(f"date range: {rainfall['date'].min():%Y-%m-%d} to {rainfall['date'].max():%Y-%m-%d}")
print(f"districts ({rainfall['district'].nunique()}): {', '.join(sorted(rainfall['district'].unique()))}")
print(f"nulls in rainfall columns: {int(rainfall[RAIN_COLUMNS].isna().sum().sum())}")
print(f"negative rainfall values: {int((rainfall[RAIN_COLUMNS] < 0).sum().sum())}")

### Findings — coverage

**The first file is misnamed.** `chirps_daily_districts_1996_2000.csv` contains
data from **1998-01-01**, not 1996 — there are no 1996 or 1997 records anywhere in
the extract. The combined record is therefore **1998-01-01 to 2025-12-31**
(10,227 days), not 1996–2025. Worth knowing before quoting a study period: two
years of the intended range were never extracted.

Otherwise the data is clean: no nulls, no negative values, and all nine districts
present in every file.

## 2. The duplicated MBALE records

Every date appears **twice** for MBALE and once for every other district. This is
an extraction artefact: the Uganda boundary layer used for zonal statistics
contains two features named MBALE — **Mbale Municipality** and the general
**Mbale District** — and the municipality lies *inside* the district.

The district's zonal mean therefore already includes the municipality's grid
cells. Keeping both would count the urban core twice, and averaging them would
produce a figure that is neither the district mean nor the municipality mean, so
**the municipality rows are dropped** and the district rows kept.

### Identifying which is which

The two series are not labelled, so which row is the district has to be
established from the data. Nested zonal statistics give a strict guarantee: if
the municipality's cells are a subset of the district's, then

```
district rain_max >= municipality rain_max    AND
district rain_min <= municipality rain_min
```

on **every** date — the larger polygon must bracket the smaller one's range. So
the district is identifiable as the row whose `[rain_min, rain_max]` interval
contains the other's.

Testing this against the data confirms the nesting and settles the identification.
It does not, however, resolve every date on its own: on a minority of days the two
polygons share the same min and max while their means differ, because the same
extreme values can appear in both while the cell mixes differ. The rule below
therefore uses the provable dates to determine **which row position** is the
district, then applies that position consistently — rather than deciding date by
date and leaving the ambiguous ones to a coin flip.

In [ ]:
def find_duplicated_districts(rainfall: pd.DataFrame) -> list[str]:
    """Districts appearing more than once on the same date."""
    counts = rainfall.groupby(["date", "district"]).size()
    return sorted(counts[counts > 1].index.get_level_values("district").unique())


def brackets(outer: pd.DataFrame, inner: pd.DataFrame) -> pd.Series:
    """True where `outer`'s rainfall range contains `inner`'s."""
    return (outer["rain_max"] >= inner["rain_max"]) & (outer["rain_min"] <= inner["rain_min"])


duplicated = find_duplicated_districts(rainfall)
print(f"districts with duplicate dates: {duplicated}")

# Pair the two series by their position within each date, preserving file order
mbale = rainfall[rainfall["district"] == "MBALE"].sort_values(["date", "source_file", "file_row"]).copy()
mbale["position"] = mbale.groupby("date").cumcount()
if set(mbale["position"].unique()) != {0, 1}:
    raise ValueError(f"expected exactly 2 MBALE rows per date, saw {sorted(mbale['position'].unique())}")

first = mbale[mbale["position"] == 0].set_index("date")
second = mbale[mbale["position"] == 1].set_index("date")

first_brackets = brackets(first, second)
second_brackets = brackets(second, first)
# A tie (identical min and max) satisfies both directions and cannot discriminate
tied = first_brackets & second_brackets
decisive = len(first) - int(tied.sum())

print(f"\n{len(first):,} dates with two MBALE rows")
print(f"  position 0 brackets position 1: {int(first_brackets.sum()):>6} ({100 * first_brackets.mean():.1f}%)")
print(f"  position 1 brackets position 0: {int(second_brackets.sum()):>6} ({100 * second_brackets.mean():.1f}%)")
print(f"  ties, identical min and max:    {int(tied.sum()):>6}")
print(f"  decisive dates:                 {decisive:>6}")

# On the decisive dates, one position must win every single time for the nesting
# claim to hold and for a positional rule to be safe
wins = {
    0: int((first_brackets & ~tied).sum()),
    1: int((second_brackets & ~tied).sum()),
}
print(f"\ndecisive dates won by position 0: {wins[0]:,}  by position 1: {wins[1]:,}")

district_position = max(wins, key=wins.get)
if wins[district_position] != decisive:
    raise ValueError(
        f"positional rule is unsafe: position {district_position} brackets on only "
        f"{wins[district_position]} of {decisive} decisive dates"
    )

print(f"\nposition {district_position} brackets the other on ALL {decisive:,} decisive dates")
print(f"-> position {district_position} is the general Mbale District; "
      f"position {1 - district_position} is Mbale Municipality")

In [ ]:
# Spread and extremes of the two series, as a sanity check against the other
# districts: a small urban polygon covers few grid cells and so varies less
summary = []
for position, label in [(district_position, "Mbale District"),
                        (1 - district_position, "Mbale Municipality")]:
    series = mbale[mbale["position"] == position]
    summary.append({
        "series": label,
        "mean_mm": series["rain_mean"].mean(),
        "mean_spread_mm": (series["rain_max"] - series["rain_min"]).mean(),
        "highest_max_mm": series["rain_max"].max(),
    })
for name, series in rainfall[rainfall["district"] != "MBALE"].groupby("district"):
    summary.append({
        "series": name.title(),
        "mean_mm": series["rain_mean"].mean(),
        "mean_spread_mm": (series["rain_max"] - series["rain_min"]).mean(),
        "highest_max_mm": series["rain_max"].max(),
    })

print("within-polygon spread, Mbale series against the other districts:")
print(pd.DataFrame(summary).round(2).to_string(index=False))

### Findings — MBALE identification

**The nesting is confirmed exactly.** Across 10,227 dates, position 0 brackets
position 1 on **all 9,417 decisive dates** — 100%, with no exceptions. The
remaining 810 dates are ties where both polygons report the same minimum and
maximum, which satisfies the test in both directions and so cannot discriminate;
658 of those are days with no rain anywhere in the district.

That unanimity is what makes the positional rule safe, and the cell above raises
an error rather than guessing if a future re-extraction breaks it.

The spread check corroborates it independently:

| Series | Mean rainfall | Mean spread | Highest max |
| --- | --- | --- | --- |
| **Mbale District** | 4.31 mm | **7.93 mm** | 91.4 mm |
| Mbale Municipality | 4.26 mm | **2.89 mm** | 79.4 mm |
| Other districts | 3.9–5.7 mm | 6.56–10.99 mm | — |

The district's within-polygon spread (7.93 mm) sits squarely inside the range of
its neighbours (6.56–10.99 mm), while the municipality's (2.89 mm) is less than
half the lowest of them — exactly what a small, few-celled urban polygon should
look like, and a far starker separation than the means alone suggest.

**Note on pairing.** These two series are only separable if the rows are paired in
their original file order. Sorting the MBALE rows by date alone is not enough:
pandas' default sort is not stable, so equal dates get reordered and the two
series are shuffled together. Doing that drops the bracketing test from 100% to
79% and blurs the spreads towards each other (6.8 against 4.0). The cell above
therefore sorts by `["date", "source_file", "file_row"]`, and the assertion on
the bracketing test is what catches the mistake if it recurs.

The two series correlate at 0.952 but differ by **1.08 mm on an average day**, so
this is not a cosmetic choice: picking the wrong series, or averaging them, would
shift daily rainfall by around a millimetre in the district that carries the most
flood records.

## 3. Merge and write

The municipality rows are dropped, leaving one row per district-day. No values are
combined, so `rain_mean`, `rain_max` and `rain_min` all keep their original
meaning — an average of two maxima would not have been a maximum of anything.

In [ ]:
municipality_rows = mbale[mbale["position"] != district_position]

before = len(rainfall)
rainfall = rainfall.drop(index=municipality_rows.index)
rainfall = (
    rainfall.drop(columns=["source_file", "file_row"])
    .sort_values(["district", "date"])
    .reset_index(drop=True)
)

print(f"dropped {len(municipality_rows):,} Mbale Municipality rows: {before:,} -> {len(rainfall):,}")

# The merged file must be a complete, gap-free daily panel with no duplicates
expected_days = pd.date_range(rainfall["date"].min(), rainfall["date"].max(), freq="D")
per_district = rainfall.groupby("district")["date"].agg(["count", "nunique"])
missing = {
    name: len(set(expected_days) - set(group["date"]))
    for name, group in rainfall.groupby("district")
}

print(f"\nrows: {len(rainfall):,}  districts: {rainfall['district'].nunique()}  "
      f"days per district expected: {len(expected_days):,}")
print(f"duplicate district-days: {int(rainfall.duplicated(['date', 'district']).sum())}")
incomplete = {name: gaps for name, gaps in missing.items() if gaps}
print(f"districts with missing days: {incomplete or 'none'}")
print(f"remaining duplicated districts: {find_duplicated_districts(rainfall) or 'none'}")
print()
print(per_district.to_string())

In [ ]:
OUT = REPO_ROOT / OUTPUT_PATH
rainfall.to_csv(OUT, index=False)

print(f"wrote {len(rainfall):,} rows to {OUT}")
print(f"  columns: {', '.join(rainfall.columns)}")
print(f"  {rainfall['date'].min():%Y-%m-%d} to {rainfall['date'].max():%Y-%m-%d}")
print()
print(rainfall.head(5).to_string(index=False))

## 4. Result

`dataset/chirps_daily_rainfall.csv` holds **92,043 rows** — 10,227 consecutive
days across 9 districts, one row per district-day, with no duplicates and no
missing days.

| Stage | Rows |
| --- | --- |
| Six extracts loaded | 102,270 |
| Mbale Municipality dropped | **92,043** |

Columns are unchanged from the extraction: `date`, `district`, `rain_mean`,
`rain_max`, `rain_min`, in mm/day.

Two things to carry forward:

1. **The record starts in 1998, not 1996.** The first file's name is wrong; there
   is no 1996 or 1997 data. Since the flood labels run to 2018-06-20, the usable
   overlap for training is **1998–2018**, about 21 years.
2. **Mbale is now the district polygon only.** Rainfall for Mbale is the zonal
   mean over the whole district, which includes the municipality. If a future
   analysis wants the urban core specifically, it must be re-extracted — the
   municipality series is not preserved in this file.